<a href="https://colab.research.google.com/github/taselshambakey/DECI-final-project/blob/main/Tasneem_DECI_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

In [1]:
import sqlite3
import json


conn = sqlite3.connect('database.db')
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# Dedup view: 8 checkout_ids appear as exact duplicate rows in the raw table.
# We collapse them so no checkout is double-counted.
cur.execute("DROP VIEW IF EXISTS checkouts_clean")
cur.execute("""
    CREATE VIEW checkouts_clean AS
    SELECT DISTINCT checkout_id, member_id, book_id, checkout_date, return_date
    FROM checkouts
""")

output_lines = []

def log(line=""):
    print(line)
    output_lines.append(str(line))


## Task 1- Goal 1

In [2]:
log("################ Q1: Checkouts per member (incl. zero) ################")
log("Reasoning:")
log("  - Every member must appear in the result, even those with no checkouts,")
log("    so members is the driving table with a LEFT JOIN out to checkouts_clean")
log("    (an INNER JOIN would silently drop members with zero checkouts).")
log("  - COUNT(c.checkout_id) counts only matched checkout rows per member; for")
log("    members with no matches the LEFT JOIN produces NULLs, which COUNT(...)")
log("    correctly reports as 0 rather than NULL.")
log("  - checkouts_clean (de-duplicated) is used so no member's total is inflated")
log("    by an accidental duplicate row.")
log("  - Sorted busiest-first (checkout_count DESC), with member_id as a")
log("    tiebreaker for a stable, reproducible order.")
log()
q1 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
LEFT JOIN checkouts_clean c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC;
"""
cur.execute(q1)
rows1 = cur.fetchall()
for r in rows1:
    log(dict(r))
log(f"Total members: {len(rows1)}")
log(f"Members with 0 checkouts: {sum(1 for r in rows1 if r['checkout_count'] == 0)}")

# ---------------------------------------------------------------------------
log()
log("################ Q2: Author pattern search ################")
log("Reasoning:")
log("  - Chosen pattern: author's first name starts with 'A' (SQL LIKE 'A%').")
log("    This is an arbitrary but concrete pattern choice, picked because it")
log("    returns a small, easy-to-verify set (3 distinct authors) rather than")
log("    an empty or overwhelming result.")
log("  - Query hits the books table only, since author is a book attribute,")
log("    not a checkout attribute -- no join to checkouts is needed.")
log("  - Sorted by author then title so same-author books are grouped together.")
log()
pattern = 'A%'
q2 = """
SELECT book_id, title, author
FROM books
WHERE author LIKE ?
ORDER BY author, title;
"""
cur.execute(q2, (pattern,))
rows2 = cur.fetchall()
for r in rows2:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q3: Top 5 most-borrowed titles ################")
log("Reasoning:")
log("  - 'Most popular' = highest raw checkout frequency per title, so we")
log("    JOIN checkouts_clean to books on book_id and COUNT checkouts per book.")
log("    An INNER JOIN is correct here (not LEFT JOIN) because a book with")
log("    zero checkouts isn't a 'popular book' and shouldn't appear at all.")
log("  - checkouts_clean (de-duplicated) is used so a duplicated checkout row")
log("    doesn't inflate a title's popularity count.")
log("  - ORDER BY times_borrowed DESC, then title ASC as a tiebreaker, then")
log("    LIMIT 5 gives exactly the top five, with a stable, reproducible order")
log("    when counts tie.")
log()
q3 = """
SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_borrowed
FROM checkouts_clean c
JOIN books b ON b.book_id = c.book_id
GROUP BY b.book_id, b.title, b.author
ORDER BY times_borrowed DESC, b.title ASC
LIMIT 5;
"""
cur.execute(q3)
rows3 = cur.fetchall()
for r in rows3:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q4: Top 10 most active readers ################")
log("Reasoning:")
log("  - Same shape as Q1 (per-member checkout count), but here we only want")
log("    members who have actually borrowed something, so an INNER JOIN")
log("    (members to checkouts_clean) is used instead of a LEFT JOIN -- a")
log("    member with 0 checkouts can't be one of the 'most active readers'.")
log("  - checkouts_clean is used again to avoid duplicate-row inflation.")
log("  - ORDER BY checkout_count DESC with member_id as a tiebreaker, then")
log("    LIMIT 10, gives the requested top-10 ranking highest to lowest.")
log()
q4 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
JOIN checkouts_clean c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC
LIMIT 10;
"""
cur.execute(q4)
rows4 = cur.fetchall()
for r in rows4:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q5: Neighborhood activity, skipping 10 most recent ################")
log("Reasoning:")
log("  - Chosen neighborhood: Maadi -- the largest neighborhood by member count,")
log("    which gives a big enough checkout history to demonstrate 'looking")
log("    further back in time'.")
log("  - The raw neighborhood column has inconsistent casing/whitespace")
log("    ('Maadi', 'Maadi ' with a trailing space, etc.), so the WHERE clause")
log("    compares LOWER(TRIM(m.neighborhood)) against a lowercase literal.")
log("    This only affects how rows are MATCHED for this query -- it does not")
log("    change or clean any stored data, so it isn't a data-cleaning step.")
log("  - 'Newest to oldest' means ORDER BY checkout_date DESC (checkout_id DESC")
log("    as a tiebreaker for same-day checkouts, for a stable order).")
log("  - 'Looking past the ten most recent' means skipping the first 10 rows")
log("    of that ordered result, done with OFFSET 10 (LIMIT -1 in SQLite means")
log("    'no limit', so this returns everything after the 10 most recent).")
log("  - checkouts_clean is used so the one duplicated Maadi row doesn't appear")
log("    twice in the output or inflate the total count.")
log()
neighborhood = 'maadi'
q5 = """
SELECT c.checkout_id, m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       c.book_id, c.checkout_date, c.return_date
FROM checkouts_clean c
JOIN members m ON m.member_id = c.member_id
WHERE LOWER(TRIM(m.neighborhood)) = ?
ORDER BY c.checkout_date DESC, c.checkout_id DESC
LIMIT -1 OFFSET 10;
"""
cur.execute(q5, (neighborhood,))
rows5 = cur.fetchall()
for r in rows5:
    log(dict(r))
log(f"Total Maadi checkouts: {len(rows5) + 10}  | shown (beyond most recent 10): {len(rows5)}")

conn.close()

# Save all answers AND their reasoning to a plain text file
output_path = "answers.txt"
with open(output_path, "w") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved answers and reasoning to {output_path}")

################ Q1: Checkouts per member (incl. zero) ################
Reasoning:
  - Every member must appear in the result, even those with no checkouts,
    so members is the driving table with a LEFT JOIN out to checkouts_clean
    (an INNER JOIN would silently drop members with zero checkouts).
  - COUNT(c.checkout_id) counts only matched checkout rows per member; for
    members with no matches the LEFT JOIN produces NULLs, which COUNT(...)
    correctly reports as 0 rather than NULL.
  - checkouts_clean (de-duplicated) is used so no member's total is inflated
    by an accidental duplicate row.
  - Sorted busiest-first (checkout_count DESC), with member_id as a
    tiebreaker for a stable, reproducible order.

{'member_id': 1034, 'member_name': 'Aya Wahba', 'checkout_count': 23}
{'member_id': 1044, 'member_name': 'Sherif Saleh', 'checkout_count': 20}
{'member_id': 1008, 'member_name': 'Ziad Saleh', 'checkout_count': 19}
{'member_id': 1010, 'member_name': 'Nour Nabil', 'checkout

## Task 1- Goal 2

Stage 1 — Members and Checkouts
================================
Goal: link every checkout to the member who made it (no checkout lost, none
mis-linked), and make it possible to see, per member, how many books they've
borrowed in total.

Constraint for this stage: pure Python only — no SQL JOIN, no pandas.merge(),
no other "query tool" doing the linking for us. We fetch the two raw tables
with plain SELECT * statements and do the matching ourselves with a
dictionary keyed by member_id.

In [3]:
DB_PATH = "database.db"

def fetch_all_as_dicts(cursor, table_name):
    """Read an entire table with a bare SELECT * (no JOIN) and return a list
    of plain dicts, one per row."""
    cursor.execute(f"SELECT * FROM {table_name}")
    columns = [d[0] for d in cursor.description]
    return [dict(zip(columns, row)) for row in cursor.fetchall()]


def dedupe_checkouts(raw_checkouts):
    """The raw checkouts table has 8 checkout_ids stored twice as identical
    rows. Collapse them in plain Python (no DISTINCT/SQL) by keeping the
    first occurrence of each checkout_id."""
    seen = set()
    deduped = []
    for row in raw_checkouts:
        cid = row["checkout_id"]
        if cid in seen:
            continue
        seen.add(cid)
        deduped.append(row)
    return deduped


def build_stage1(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    raw_members = fetch_all_as_dicts(cur, "members")
    raw_checkouts = fetch_all_as_dicts(cur, "checkouts")
    conn.close()

    checkouts = dedupe_checkouts(raw_checkouts)

    # Index members by member_id for O(1) lookup -- this dict IS the "join",
    # done by hand instead of by a query engine.
    members_by_id = {m["member_id"]: m for m in raw_members}

    combined = []
    unmatched_checkouts = []          # checkouts whose member_id has no member record
    checkout_counts = {m["member_id"]: 0 for m in raw_members}  # every member starts at 0

    for c in checkouts:
        member = members_by_id.get(c["member_id"])
        if member is None:
            # A checkout referencing a member that doesn't exist would be a
            # mis-link risk; we flag it instead of silently guessing.
            unmatched_checkouts.append(c)
            continue

        row = {
            "checkout_id": c["checkout_id"],
            "source": "database",
            "member_id": member["member_id"],
            "first_name": member["first_name"],
            "last_name": member["last_name"],
            "member_name": f"{member['first_name']} {member['last_name']}",
            "grade": member["grade"],
            "neighborhood": member["neighborhood"],
            "membership_status": member["membership_status"],
            "join_date": member["join_date"],
            "book_id": c["book_id"],
            "checkout_date": c["checkout_date"],
            "return_date": c["return_date"],
        }
        combined.append(row)
        checkout_counts[member["member_id"]] += 1

    # Attach each member's running total onto their own rows, so the total is
    # visible directly on the combined data without a second lookup.
    for row in combined:
        row["member_total_checkouts"] = checkout_counts[row["member_id"]]

    return {
        "combined": combined,
        "checkout_counts": checkout_counts,   # includes members with 0 checkouts
        "members_by_id": members_by_id,
        "raw_checkout_count": len(raw_checkouts),
        "deduped_checkout_count": len(checkouts),
        "unmatched_checkouts": unmatched_checkouts,
    }

result = build_stage1()
combined = result["combined"]

print(f"Raw checkout rows read: {result['raw_checkout_count']}")
print(f"Checkout rows after de-duplication: {result['deduped_checkout_count']}")
print(f"Checkout rows successfully linked to a member: {len(combined)}")
print(f"Checkouts that could not be matched to a member: {len(result['unmatched_checkouts'])}")
print(f"Members represented (incl. members with 0 checkouts): {len(result['checkout_counts'])}")
print()
print("Sample combined rows:")
for row in combined[:3]:
    print(row)
print()
print("Per-member totals (first 5, sorted by member_id):")
for member_id in sorted(result["checkout_counts"])[:5]:
    print(f"  member_id {member_id}: {result['checkout_counts'][member_id]} checkouts")

Raw checkout rows read: 391
Checkout rows after de-duplication: 383
Checkout rows successfully linked to a member: 383
Checkouts that could not be matched to a member: 0
Members represented (incl. members with 0 checkouts): 80

Sample combined rows:
{'checkout_id': 9263, 'source': 'database', 'member_id': 1047, 'first_name': 'Sara', 'last_name': 'Rashad', 'member_name': 'Sara Rashad', 'grade': None, 'neighborhood': 'Heliopolis', 'membership_status': 'Inactive', 'join_date': '2024-06-25', 'book_id': 517, 'checkout_date': '2024-10-21', 'return_date': '2024-11-07', 'member_total_checkouts': 16}
{'checkout_id': 9340, 'source': 'database', 'member_id': 1072, 'first_name': 'Seif', 'last_name': 'Zaki', 'member_name': 'Seif Zaki', 'grade': 9, 'neighborhood': 'Zamalek', 'membership_status': 'Active', 'join_date': '2025-10-21', 'book_id': 513, 'checkout_date': '2025-08-24', 'return_date': '2025-09-01', 'member_total_checkouts': 14}
{'checkout_id': 9231, 'source': 'database', 'member_id': 1053, '

Stage 2 — Book Details
=======================
Goal: attach each checkout's book details so no checkout loses its book
info and none ends up with the wrong book's details. The checkout ROW COUNT
must stay exactly the same as Stage 1 -- this stage adds columns, not rows.

Book details live in two places that both need to be combined first:
  - database.db -> books table: book_id, title, author
  - books.json          : book_id, genre, pages, publication_year, publisher
Both are keyed by book_id (501-532), so they're merged into one lookup dict
before being attached to each checkout.

In [4]:
BOOKS_JSON_PATH = "books.json"

def load_books_catalog(db_path=DB_PATH, json_path=BOOKS_JSON_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    db_books = fetch_all_as_dicts(cur, "books")   # book_id, title, author
    conn.close()

    with open(json_path) as f:
        json_books = json.load(f)                # book_id, genre, pages, publication_year, publisher

    db_books_by_id = {b["book_id"]: b for b in db_books}
    json_books_by_id = {b["book_id"]: b for b in json_books}

    all_ids = set(db_books_by_id) | set(json_books_by_id)
    only_in_db = set(db_books_by_id) - set(json_books_by_id)
    only_in_json = set(json_books_by_id) - set(db_books_by_id)

    books_by_id = {}
    for book_id in all_ids:
        merged = {}
        merged.update(db_books_by_id.get(book_id, {}))
        merged.update(json_books_by_id.get(book_id, {}))
        books_by_id[book_id] = merged

    return {
        "books_by_id": books_by_id,
        "only_in_db": only_in_db,
        "only_in_json": only_in_json,
    }


def build_stage2():
    stage1_result = build_stage1()
    stage1_rows = stage1_result["combined"]

    catalog = load_books_catalog()
    books_by_id = catalog["books_by_id"]

    stage2_rows = []
    unmatched = []

    for row in stage1_rows:
        book = books_by_id.get(row["book_id"])
        new_row = dict(row)  # copy -- don't mutate stage1 data
        if book is None:
            unmatched.append(row["checkout_id"])
            new_row.update({
                "title": None, "author": None, "genre": None,
                "pages": None, "publication_year": None, "publisher": None,
            })
        else:
            new_row.update({
                "title": book.get("title"),
                "author": book.get("author"),
                "genre": book.get("genre"),
                "pages": book.get("pages"),
                "publication_year": book.get("publication_year"),
                "publisher": book.get("publisher"),
            })
        stage2_rows.append(new_row)

    return {
        "combined": stage2_rows,
        "stage1_row_count": len(stage1_rows),
        "stage2_row_count": len(stage2_rows),
        "unmatched_checkout_ids": unmatched,
        "catalog_only_in_db": catalog["only_in_db"],
        "catalog_only_in_json": catalog["only_in_json"],
    }

result = build_stage2()

print(f"Stage 1 row count: {result['stage1_row_count']}")
print(f"Stage 2 row count: {result['stage2_row_count']}")
assert result["stage1_row_count"] == result["stage2_row_count"], \
    "Row count changed during Stage 2 -- this should never happen!"
print("Row count unchanged: PASS")
print(f"Checkouts whose book_id had no catalog match: {len(result['unmatched_checkout_ids'])}")
print(f"Book IDs present only in the DB books table: {sorted(result['catalog_only_in_db'])}")
print(f"Book IDs present only in books.json: {sorted(result['catalog_only_in_json'])}")
print()
print("Sample combined rows:")
for row in result["combined"][:3]:
    print(row)

Stage 1 row count: 383
Stage 2 row count: 383
Row count unchanged: PASS
Checkouts whose book_id had no catalog match: 0
Book IDs present only in the DB books table: []
Book IDs present only in books.json: []

Sample combined rows:
{'checkout_id': 9263, 'source': 'database', 'member_id': 1047, 'first_name': 'Sara', 'last_name': 'Rashad', 'member_name': 'Sara Rashad', 'grade': None, 'neighborhood': 'Heliopolis', 'membership_status': 'Inactive', 'join_date': '2024-06-25', 'book_id': 517, 'checkout_date': '2024-10-21', 'return_date': '2024-11-07', 'member_total_checkouts': 16, 'title': 'Shadows on the Corniche', 'author': 'Hani Nagati', 'genre': 'Mystery', 'pages': 338, 'publication_year': 2015, 'publisher': 'Delta House'}
{'checkout_id': 9340, 'source': 'database', 'member_id': 1072, 'first_name': 'Seif', 'last_name': 'Zaki', 'member_name': 'Seif Zaki', 'grade': 9, 'neighborhood': 'Zamalek', 'membership_status': 'Active', 'join_date': '2025-10-21', 'book_id': 513, 'checkout_date': '2025-0